In [2]:
import os
import re
import requests
import sys
import json
from dotenv import load_dotenv
load_dotenv("credentials.env", override=True)

True

In [3]:
# Example function for extracting information from healthcare-related text 
def health_example(client, documents):
    
    #Patient needs to take 50 mg of ibuprofen.
    poller = client.begin_analyze_healthcare_entities(documents)
    result = poller.result()

    docs = [doc for doc in result if not doc.is_error]

    for idx, doc in enumerate(docs):
        for entity in doc.entities:
            print("Entity: {}".format(entity.text))
            print("...Normalized Text: {}".format(entity.normalized_text))
            print("...Category: {}".format(entity.category))
            print("...Subcategory: {}".format(entity.subcategory))
            print("...Offset: {}".format(entity.offset))
            print("...Confidence score: {}".format(entity.confidence_score))
        for relation in doc.entity_relations:
            print("Relation of type: {} has the following roles".format(relation.relation_type))
            for role in relation.roles:
                print("...Role '{}' with entity '{}'".format(role.name, role.entity.text))
        print("------------------------------------------")

    return docs

In [4]:

# This example requires environment variables named "LANGUAGE_KEY" and "LANGUAGE_ENDPOINT"
key = os.environ.get('LANGUAGE_KEY')
endpoint = os.environ.get('LANGUAGE_ENDPOINT')

from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

# Authenticate the client using your key and endpoint 
def authenticate_client():
    ta_credential = AzureKeyCredential(key)
    text_analytics_client = TextAnalyticsClient(
            endpoint=endpoint, 
            credential=ta_credential)
    return text_analytics_client

client = authenticate_client()
documents = [
        """        
        Patient visited for 99213, was diagnosed R06.2, E11.65. Type 2 diabetes, wheezing.
        """
    ]
result = health_example(client, documents)



Entity: visited
...Normalized Text: None
...Category: AdministrativeEvent
...Subcategory: None
...Offset: 25
...Confidence score: 0.57
Entity: 99213
...Normalized Text: None
...Category: Time
...Subcategory: None
...Offset: 37
...Confidence score: 0.68
Entity: R06.2
...Normalized Text: None
...Category: Variant
...Subcategory: None
...Offset: 58
...Confidence score: 0.83
Entity: E11.65
...Normalized Text: None
...Category: Variant
...Subcategory: None
...Offset: 65
...Confidence score: 0.66
Entity: Type 2 diabetes
...Normalized Text: Diabetes Mellitus, Non-Insulin-Dependent
...Category: Diagnosis
...Subcategory: None
...Offset: 73
...Confidence score: 1.0
Entity: wheezing
...Normalized Text: Wheezing
...Category: SymptomOrSign
...Subcategory: None
...Offset: 90
...Confidence score: 0.98
Relation of type: TimeOfEvent has the following roles
...Role 'Event' with entity 'visited'
...Role 'Time' with entity '99213'
------------------------------------------


In [5]:
for idx, doc in enumerate(result):
        for entity in doc.entities:
            print("Entity: {}".format(entity))

Entity: {'text': 'visited', 'normalized_text': None, 'category': 'AdministrativeEvent', 'subcategory': None, 'assertion': None, 'length': 7, 'offset': 25, 'confidence_score': 0.57, 'data_sources': None}
Entity: {'text': '99213', 'normalized_text': None, 'category': 'Time', 'subcategory': None, 'assertion': None, 'length': 5, 'offset': 37, 'confidence_score': 0.68, 'data_sources': None}
Entity: {'text': 'R06.2', 'normalized_text': None, 'category': 'Variant', 'subcategory': None, 'assertion': None, 'length': 5, 'offset': 58, 'confidence_score': 0.83, 'data_sources': None}
Entity: {'text': 'E11.65', 'normalized_text': None, 'category': 'Variant', 'subcategory': None, 'assertion': None, 'length': 6, 'offset': 65, 'confidence_score': 0.66, 'data_sources': None}
Entity: {'text': 'Type 2 diabetes', 'normalized_text': 'Diabetes Mellitus, Non-Insulin-Dependent', 'category': 'Diagnosis', 'subcategory': None, 'assertion': None, 'length': 15, 'offset': 73, 'confidence_score': 1.0, 'data_sources':

In [17]:
def convert_to_json(entity):
    # Convert the result to JSON format
    data = {
        "documents": [
            {            
                "text": entity.text,
                "normalized_text": entity.normalized_text,
                "category": entity.category,
                "subcategory": entity.subcategory,
            }
            for entity in result[0].entities
        ]
    }
    return json.dumps(data, indent=4)

convert_to_json(result[0].entities[0])
# data = {
#     "documents": [
#         {            
#             "text": result[0].entities[0].text,
#             "normalized_text": result[0].entities[0].normalized_text,
#             "category": result[0].entities[0].category,
#             "subcategory": result[0].entities[0].subcategory,
#         }
#     ]
# }
# json.dumps(data)
# result[0].entities[0].text
# result[0].entities[0].normalized_text

'{\n    "documents": [\n        {\n            "text": "visited",\n            "normalized_text": null,\n            "category": "AdministrativeEvent",\n            "subcategory": null\n        },\n        {\n            "text": "99213",\n            "normalized_text": null,\n            "category": "Time",\n            "subcategory": null\n        },\n        {\n            "text": "R06.2",\n            "normalized_text": null,\n            "category": "Variant",\n            "subcategory": null\n        },\n        {\n            "text": "E11.65",\n            "normalized_text": null,\n            "category": "Variant",\n            "subcategory": null\n        },\n        {\n            "text": "Type 2 diabetes",\n            "normalized_text": "Diabetes Mellitus, Non-Insulin-Dependent",\n            "category": "Diagnosis",\n            "subcategory": null\n        },\n        {\n            "text": "wheezing",\n            "normalized_text": "Wheezing",\n            "category": "S

In [7]:
umls_api_key = os.environ.get('UMLS_API_KEY')
umls_uri = f"https://uts-ws.nlm.nih.gov/rest/content/2015AB/source/SNOMEDCT_US/13644009?apiKey={umls_api_key}"


# Set up the request headers with authentication
headers = {
    'Accept': 'application/json'
}
response = requests.get(umls_uri, headers=headers)
response.json()


{'pageSize': 25,
 'pageNumber': 1,
 'pageCount': 1,
 'result': {'classType': 'SourceAtomCluster',
  'ui': '13644009',
  'suppressible': False,
  'obsolete': False,
  'rootSource': 'SNOMEDCT_US',
  'atomCount': 6,
  'cVMemberCount': 0,
  'attributes': 'https://uts-ws.nlm.nih.gov/rest/content/2015AB/source/SNOMEDCT_US/13644009/attributes',
  'atoms': 'https://uts-ws.nlm.nih.gov/rest/content/2015AB/source/SNOMEDCT_US/13644009/atoms',
  'ancestors': 'https://uts-ws.nlm.nih.gov/rest/content/2015AB/source/SNOMEDCT_US/13644009/ancestors',
  'parents': 'https://uts-ws.nlm.nih.gov/rest/content/2015AB/source/SNOMEDCT_US/13644009/parents',
  'children': 'https://uts-ws.nlm.nih.gov/rest/content/2015AB/source/SNOMEDCT_US/13644009/children',
  'descendants': 'https://uts-ws.nlm.nih.gov/rest/content/2015AB/source/SNOMEDCT_US/13644009/descendants',
  'relations': 'https://uts-ws.nlm.nih.gov/rest/content/2015AB/source/SNOMEDCT_US/13644009/relations',
  'definitions': 'NONE',
  'concepts': 'https://uts-

In [8]:
def get_weather(location):
    return "sunny" 


In [9]:
from openai import AzureOpenAI


client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"),
  api_key = os.getenv("AZURE_OPENAI_API_KEY"),
  api_version="2025-03-01-preview"
)

response = client.responses.create(  
    model="gpt-4o",  # replace with your model deployment name  
    tools=[  
        {  
            "type": "function",  
            "name": "get_weather",  
            "description": "Get the weather for a location",  
            "parameters": {  
                "type": "object",  
                "properties": {  
                    "location": {"type": "string"},  
                },  
                "required": ["location"],  
            },  
        }  
    ],  
    input=[{"role": "user", "content": "What's the weather in San Francisco?"}],  
)  

print(response.model_dump_json(indent=2))  
  
# To provide output to tools, add a response for each tool call to an array passed  
# to the next response as `input`  
input = []  
for output in response.output:  
    if output.type == "function_call":  
        match output.name:  
            case "get_weather":  
                input.append(  
                    {  
                        "type": "function_call_output",  
                        "call_id": output.call_id,  
                        "output": '{"temperature": "70 degrees"}',  
                    }  
                )  
            case _:  
                raise ValueError(f"Unknown function call: {output.name}")  
  
second_response = client.responses.create(  
    model="gpt-4o",  
    previous_response_id=response.id,  
    input=input  
)  

print(second_response.model_dump_json(indent=2))

APIStatusError: Error code: 424 - {'error': {'message': 'Invalid URL (POST /v1/responses)', 'type': 'invalid_request_error', 'param': None, 'code': None}}